# Workforce Planning Analysis

## Project Overview
Analyze staffing levels, demand, turnover, and hiring timelines to support workforce planning decisions.

## Learning Objectives
- Calculate staffing gaps between demand and supply
- Analyze turnover impact on workforce capacity
- Model hiring pipeline capacity and time-to-fill
- Identify departments with chronic understaffing

## Problem Statement
Operations and HR need to forecast staffing needs, understand where gaps exist, and plan hiring to maintain operational capacity across departments.

## Why This Project Matters
Poor workforce planning leads to understaffing (burnout, missed targets) or overstaffing (wasted budget). Data-driven planning optimizes the balance.

## Dataset Overview
Synthetic workforce planning dataset: monthly staffing snapshots for 8 departments over 24 months with demand, headcount, turnover, and hiring data.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
departments = ['Engineering', 'Sales', 'Operations', 'Support', 'Marketing', 'Finance', 'Product', 'HR']
months = pd.date_range('2023-01-01', '2024-12-01', freq='MS')
rows = []
base_demand = {'Engineering': 120, 'Sales': 90, 'Operations': 75, 'Support': 60,
               'Marketing': 45, 'Finance': 35, 'Product': 40, 'HR': 25}
for dept in departments:
    demand = base_demand[dept]
    headcount = int(demand * np.random.uniform(0.85, 1.0))
    for m in months:
        # Seasonal demand variation
        seasonal = 1 + 0.05 * np.sin(2 * np.pi * m.month / 12)
        demand_adj = int(demand * seasonal * (1 + np.random.uniform(-0.03, 0.05)))
        turnover = max(0, int(np.random.poisson(headcount * 0.015)))
        hires = max(0, int(np.random.poisson(max(0, demand_adj - headcount + turnover) * 0.6)))
        headcount = headcount - turnover + hires
        time_to_fill = np.clip(np.random.normal(35, 12), 10, 90).round(0)
        open_reqs = max(0, demand_adj - headcount)
        rows.append({
            'Department': dept, 'Month': m,
            'Demand': demand_adj, 'Headcount': headcount,
            'Turnover': turnover, 'Hires': hires,
            'OpenReqs': open_reqs, 'TimeToFill': time_to_fill,
        })
        demand = demand_adj

df = pd.DataFrame(rows)
df['StaffingGap'] = df['Demand'] - df['Headcount']
df['GapPct'] = (df['StaffingGap'] / df['Demand'] * 100).round(1)
df['TurnoverRate'] = (df['Turnover'] / df['Headcount'] * 100).round(1)

print(f'Dataset: {df.shape}')
print(f'Departments: {df["Department"].nunique()}, Months: {df["Month"].nunique()}')
df.head(10)

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nDepartment counts: {df["Department"].value_counts().to_dict()}')
print(f'Date range: {df["Month"].min()} to {df["Month"].max()}')

## Overall Staffing Gap Trend

In [ ]:
agg_monthly = df.groupby('Month').agg(
    total_demand=('Demand', 'sum'),
    total_headcount=('Headcount', 'sum'),
    total_gap=('StaffingGap', 'sum'),
    total_turnover=('Turnover', 'sum'),
    total_hires=('Hires', 'sum'),
).round(0)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(agg_monthly.index, agg_monthly['total_demand'], label='Demand', linewidth=2, color='red')
ax.plot(agg_monthly.index, agg_monthly['total_headcount'], label='Headcount', linewidth=2, color='steelblue')
ax.fill_between(agg_monthly.index, agg_monthly['total_headcount'], agg_monthly['total_demand'],
                alpha=0.2, color='coral', label='Gap')
ax.set_title('Total Workforce: Demand vs Headcount')
ax.legend()
ax.set_xlabel('Month')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'staffing_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Department Staffing Gaps

In [ ]:
dept_avg = df.groupby('Department').agg(
    avg_demand=('Demand', 'mean'),
    avg_headcount=('Headcount', 'mean'),
    avg_gap=('StaffingGap', 'mean'),
    avg_gap_pct=('GapPct', 'mean'),
    avg_turnover_rate=('TurnoverRate', 'mean'),
    avg_time_to_fill=('TimeToFill', 'mean'),
).round(1)
print('Department Staffing Summary:')
print(dept_avg.sort_values('avg_gap_pct', ascending=False))

fig, ax = plt.subplots(figsize=(10, 5))
dept_avg.sort_values('avg_gap_pct')['avg_gap_pct'].plot.barh(ax=ax, edgecolor='black', color='coral')
ax.set_title('Avg Staffing Gap % by Department')
ax.set_xlabel('Gap %')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'dept_gaps.png'), dpi=100, bbox_inches='tight')
plt.show()

## Turnover vs Hiring Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dept_flow = df.groupby('Department').agg(
    total_turnover=('Turnover', 'sum'),
    total_hires=('Hires', 'sum'),
)
dept_flow['net_flow'] = dept_flow['total_hires'] - dept_flow['total_turnover']
dept_flow.sort_values('net_flow')[['total_hires', 'total_turnover']].plot.barh(ax=axes[0], edgecolor='black')
axes[0].set_title('Total Hires vs Turnover by Department')

monthly_flow = df.groupby('Month').agg(turnover=('Turnover', 'sum'), hires=('Hires', 'sum'))
axes[1].bar(monthly_flow.index, monthly_flow['hires'], color='steelblue', alpha=0.7, label='Hires', width=20)
axes[1].bar(monthly_flow.index, -monthly_flow['turnover'], color='coral', alpha=0.7, label='Turnover', width=20)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('Monthly Hires vs Turnover')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'turnover_hiring.png'), dpi=100, bbox_inches='tight')
plt.show()

## Time-to-Fill Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for dept in ['Engineering', 'Sales', 'Support', 'Operations']:
    sub = df[df['Department'] == dept]
    ax.plot(sub['Month'], sub['TimeToFill'], marker='.', alpha=0.7, label=dept)
ax.set_title('Time-to-Fill Trend (Selected Departments)')
ax.set_ylabel('Days')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'time_to_fill.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f'Overall time-to-fill stats:\n{df["TimeToFill"].describe().round(1)}')

## Capacity Risk Score

In [ ]:
# Simple risk score: high gap% + high turnover + long time-to-fill = high risk
from sklearn.preprocessing import MinMaxScaler
risk_df = dept_avg[['avg_gap_pct', 'avg_turnover_rate', 'avg_time_to_fill']].copy()
scaler = MinMaxScaler()
scaled = pd.DataFrame(scaler.fit_transform(risk_df), index=risk_df.index, columns=risk_df.columns)
risk_df['RiskScore'] = (scaled.mean(axis=1) * 100).round(1)
print('Workforce Planning Risk Score by Department:')
print(risk_df.sort_values('RiskScore', ascending=False))

## Interpretation and Business Insight
- **Engineering and Sales** carry the largest staffing gaps — these are high-demand, competitive-market roles
- **Turnover exceeds hiring** in several departments, creating compounding gaps
- **Time-to-fill** averages ~35 days but varies widely — slow hiring in high-turnover departments worsens gaps
- **Seasonal demand patterns** (Q4 peaks) require proactive hiring 1-2 months in advance
- The **risk score** combines gap, turnover, and hiring speed to prioritize workforce planning efforts
- Departments with high turnover AND long time-to-fill are in a "staffing death spiral"—they need urgent attention

## Limitations
- Synthetic data with simplified demand/supply dynamics
- No skill-level granularity (all headcount treated equally)
- No cost modeling (salary, overtime, contractor)
- No contractor/temp worker buffer analysis
- Single-scenario planning (no what-if modeling)

## How to Improve This Project
- Add scenario planning (growth, contraction, status quo)
- Include contractor and temp workforce buffers
- Add skill-level demand planning
- Model cost impact of understaffing (overtime, lost revenue)
- Build forward-looking forecasts using time series methods

## Production Considerations
- Monthly workforce planning dashboards for HR and finance
- Automated staffing gap alerts by department
- Quarterly demand forecasting integrated with business planning
- Recruiter capacity planning based on open req volume

## Common Mistakes
- Planning headcount without accounting for turnover (need to hire above replacement)
- Not adjusting for seasonal demand patterns
- Treating all roles equally in planning (time-to-fill varies dramatically by role)
- Reactive hiring instead of proactive pipeline building

## Mini Challenge / Exercises
1. Calculate the total FTE-months lost to understaffing across all departments.
2. If time-to-fill improved by 20%, how many fewer open-req-months would result?
3. Which department has the worst turnover-to-hire ratio?
4. Build a 3-month forward staffing forecast for Engineering assuming current trends continue.

## Final Summary / Key Takeaways
- Workforce planning requires balancing demand forecasting, turnover modeling, and hiring capacity
- Staffing gaps compound over time when turnover outpaces hiring
- Department-level risk scoring helps prioritize planning resources
- Proactive, data-driven workforce planning prevents reactive firefighting
- Time-to-fill is a critical bottleneck — reducing it directly closes staffing gaps faster